# Dataset Cleaning, Parameter Selection and MMC2–MMC3 Integration

**Project:** Explainable Inhibitor Risk Classification for Haemophilia A Using Fused Genomic–Clinical Data
**Project ID:** PW26_DN_03 · **Institution:** PES University

This notebook documents **how the existing implementation prepares the data**. It covers:
- how the two source tables were inspected and cleaned;
- which parameters were kept, transformed or excluded, and why;
- how the variant table (MMC2) and the report table (MMC3) are combined into the modelling dataset.

**How it works.** The cleaning is not re-implemented here.
- The notebook imports the project's own functions (`src/preprocessing.py`, `src/fusion.py`,
  `src/labels.py`, `src/features.py`) and prints their source where a rule is explained.
- Results are taken from the saved processed files in `data/processed/` and the reports in `results/`.
- The cleaning functions are re-run **in memory only**, to confirm that they reproduce the saved files.
  Nothing on disk is modified.
- No model is trained.

> Column interpretations come from the column names, their values and the project's audit notes. The
> source article's own data documentation has **not** been available to verify them, so they are
> marked as *inferred* throughout.

In [1]:
import inspect
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

# Project root = the folder holding README.md and src/ (works from notebooks/ or the root).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "README.md").exists() and (p / "src").exists())
sys.path.insert(0, str(ROOT))

from src import data_audit, features, fusion, labels
from src import preprocessing as pp
from src.fusion import (CLINICAL_COLUMNS, FORBIDDEN_PREDICTORS, GENOMIC_COLUMNS, GROUP_KEY,
                        NON_PREDICTOR_COLUMNS, TARGET, FusionConfig)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.precision", 3)

CONFIG = FusionConfig.load(ROOT / "configs" / "data.json")
PROCESSED = ROOT / "data" / "processed"
RESULTS = ROOT / "results"


def show_source(func):
    """Print the project's own implementation of a function."""
    display(Markdown(f"```python\n{inspect.getsource(func)}```"))


print("Project root:", ROOT)
print("Raw files:", CONFIG.mmc2.relative_to(ROOT), "and", CONFIG.mmc3.relative_to(ROOT))

Project root: C:\PROJECT\asd\hemophilia-inhibitor-prediction
Raw files: data\raw\mmc2.csv and data\raw\mmc3.csv


## 1. Project objective

FVIII inhibitors are neutralising antibodies against factor VIII. They develop in some people with
haemophilia A after replacement therapy and make standard treatment ineffective. Inhibitor risk is
known to depend partly on the **F8 genotype** (for example, null mutations such as large deletions and
nonsense mutations) and partly on **clinical characteristics** (disease severity and residual FVIII
activity).

This project asks how well a model can classify **reported inhibitor status** from these two kinds of
information combined, and explains its predictions with SHAP and LIME. Combining the two blocks lets
the project test whether genomic and clinical information each add value. Phase J ablations found that
the combined set outperformed either block alone.

**The two source tables** (supplementary files of manuscript VTH-2024-000215; journal, database version
and licence **not yet verified**):

| Table | One row per | Role in the project |
|---|---|---|
| **MMC2** | F8 variant (`mut_id`) | Genomic description of each variant: type, effect, domain, position, nucleotide and amino-acid change |
| **MMC3** | Published allele/patient report (`all_id`) | Clinical description and the **inhibitor outcome** for each reported patient carrying a variant |

One variant in MMC2 can appear in many MMC3 reports, so the modelling unit is **one labelled MMC3
report** with its variant's genomic fields attached.

## 2. Original dataset inspection

The raw CSVs are read with the project loader `pp.load_raw`. It reads every cell as **text**
(`keep_default_na=False`), so tokens such as `NA` or `NULL` are not silently converted before the
cleaning rules see them. The files are only read, never written.

In [2]:
show_source(pp.load_raw)
mmc2 = pp.load_raw(CONFIG.mmc2)
mmc3 = pp.load_raw(CONFIG.mmc3)
raw = {"MMC2": mmc2, "MMC3": mmc3}
display(pd.DataFrame({name: {"rows": f.shape[0], "columns": f.shape[1],
                             "sha256 (first 16)": pp.sha256(p)[:16]}
                      for (name, f), p in zip(raw.items(), (CONFIG.mmc2, CONFIG.mmc3))}).T)

```python
def load_raw(path: Path) -> pd.DataFrame:
    """Read a CSV with every cell as an untouched string."""
    if not path.exists():
        raise FileNotFoundError(f"Raw file not found: {path}")
    return pd.read_csv(path, dtype=str, keep_default_na=False)
```

,rows,columns,sha256 (first 16)
MMC2,6211,34,d64fbd90654d329a
MMC3,10064,23,f6bb8b1e6e867292


### 2.1 Original column names

In [3]:
width = max(len(mmc2.columns), len(mmc3.columns))
display(pd.DataFrame({"MMC2": list(mmc2.columns) + [""] * (width - len(mmc2.columns)),
                      "MMC3": list(mmc3.columns) + [""] * (width - len(mmc3.columns))}))

,MMC2,MMC3
0,mut_id,mut_id
1,mut_type,all_id
2,mut_effect,pa_id
3,g_id,subgroups
4,g_name,clotting
5,d_id,discrep
6,location,ratio
7,e_i_numb,assay
8,locnumb,antigen
9,aa_numb_old,act/ant


### 2.2 Data types and missing values

Every column is stored as text. The table below shows the **inferred** type (`data_audit.infer_type`)
and the missingness.

Missingness is counted **including the sentinel tokens** the dataset uses for "no value", not only
blank cells. The sentinel list is `pp.MISSING_TOKENS`.

In [4]:
print("Sentinel tokens treated as missing:", sorted(pp.MISSING_TOKENS))


def profile(frame, name):
    missing = frame.apply(pp.is_missing)
    return pd.DataFrame({
        "dataset": name,
        "inferred_type": [data_audit.infer_type(frame[c]) for c in frame.columns],
        "n_missing": missing.sum().values,
        "pct_missing": (100 * missing.mean()).round(2).values,
        "n_unique": frame.nunique().values,
    }, index=pd.Index(frame.columns, name="column"))


prof = pd.concat([profile(mmc2, "MMC2"), profile(mmc3, "MMC3")])
display(prof)

Sentinel tokens treated as missing: ['', '-', '?', 'na', 'nan', 'none', 'not', 'not r', 'not report', 'not reported', 'null']


,dataset,inferred_type,n_missing,pct_missing,n_unique
column,,,,,
mut_id,MMC2,numeric,0,0.00,6211
mut_type,MMC2,categorical,20,0.32,13
mut_effect,MMC2,categorical,55,0.89,25
g_id,MMC2,categorical,0,0.00,2
g_name,MMC2,categorical,0,0.00,2
d_id,MMC2,categorical,27,0.43,29
location,MMC2,categorical,28,0.45,12
e_i_numb,MMC2,mixed_numeric_text,75,1.21,260
locnumb,MMC2,mixed_numeric_text,62,1.00,239


### 2.3 Duplicate records

The checks below test both tables for duplicates:
- each table's own identifier (`mut_id` in MMC2, `all_id` in MMC3);
- exact whole-row duplicates;
- MMC3 rows that are identical apart from `all_id`;
- `(ref_id, pa_id)` patient codes that appear more than once.

In [5]:
content_cols = [c for c in mmc3.columns if c != "all_id"]
pa_known = mmc3[~pp.is_missing(mmc3["pa_id"])]
dup = pd.DataFrame({
    "MMC2": {"identifier": "mut_id", "identifier unique": mmc2["mut_id"].is_unique,
             "exact duplicate rows": int(mmc2.duplicated().sum()),
             "rows identical except id": int(mmc2.drop(columns="mut_id").duplicated().sum())},
    "MMC3": {"identifier": "all_id", "identifier unique": mmc3["all_id"].is_unique,
             "exact duplicate rows": int(mmc3.duplicated().sum()),
             "rows identical except id": int(mmc3.duplicated(subset=content_cols).sum())},
}).T
display(dup)
pairs = pa_known.groupby(["ref_id", "pa_id"])["all_id"].count()
print(f"(ref_id, pa_id) pairs with a patient code: {len(pairs):,}; pairs with more than one report: "
      f"{int((pairs > 1).sum())}")
print("Distinct mut_id in MMC3:", mmc3["mut_id"].nunique(), "| reports per variant (max):",
      int(mmc3["mut_id"].value_counts().max()))

,identifier,identifier unique,exact duplicate rows,rows identical except id
MMC2,mut_id,True,0,12
MMC3,all_id,True,0,459


(ref_id, pa_id) pairs with a patient code: 2,527; pairs with more than one report: 20
Distinct mut_id in MMC3: 6212 | reports per variant (max): 137


*Interpretation.*
- Both identifiers are unique.
- MMC3 rows that are identical apart from `all_id` are **not** removed by the implementation. Each
  `all_id` is treated as a separate published report: two patients from one paper can share every
  recorded field, and the data cannot distinguish them from a duplicate entry.
- `(ref_id, pa_id)` pairs with several reports may be one patient carrying more than one variant, or a
  repeated entry. The implementation does not deduplicate these. **Flagged for review.**
- Many MMC3 reports share one `mut_id`, which is the expected one-to-many structure.

### 2.4 Descriptive statistics of the numeric fields

In [6]:
numeric_raw = pd.DataFrame({
    "MMC2 aa_numb (amino-acid position)": pp.to_number(mmc2["aa_numb"]),
    "MMC2 nuc_numb (cDNA position)": pp.to_number(mmc2["nuc_numb"]),
    "MMC2 n_bp (length, bp)": pp.to_number(mmc2["n_bp"]),
    "MMC2 Count_mut_id": pp.to_number(mmc2["Count_mut_id"]),
})
display(numeric_raw.describe().T)
fviii = pd.DataFrame(mmc3["clotting"].map(pp.parse_fviii).tolist(), columns=["value", "kind"])
print("MMC3 'clotting' (FVIII:C) after parsing - how values were reported:")
display(fviii["kind"].value_counts().to_frame("reports"))
display(fviii["value"].astype(float).describe().to_frame("FVIII:C (%)").T)

,count,mean,std,min,25%,50%,75%,max
MMC2 aa_numb (amino-acid position),5582.0,1008.619,784.759,-42.0,266.25,870.0,1786.75,2352.0
MMC2 nuc_numb (cDNA position),6033.0,3278.047,2304.258,-2174.0,1164.00,2945.0,5564.00,8527.0
"MMC2 n_bp (length, bp)",5755.0,4.061,93.217,1.0,1.00,1.0,1.00,6230.0
MMC2 Count_mut_id,6203.0,2.477,34.953,0.0,0.00,1.0,1.00,2524.0


MMC3 'clotting' (FVIII:C) after parsing - how values were reported:


,reports
kind,
missing,3889
upper_bound,3133
exact,2255
lower_bound,400
range_mid,377
unparsed,10


,count,mean,std,min,25%,50%,75%,max
FVIII:C (%),6165.0,5.771,9.385,0.2,1.0,1.0,5.0,126.0


### 2.5 Example records

The examples below leave out patient codes (`pa_id`), free-text comments, publication and laboratory
fields, and patient country. Those fields are not needed to understand the structure, and they could
point to individual reports.

In [7]:
display(mmc2[["mut_id", "g_name", "mut_type", "mut_effect", "d_id", "location", "e_i_numb", "aa_numb",
              "nuc_numb", "ntchange", "aa_first", "aa_last", "uinhibitor"]].head(4))
hidden = {"pa_id", "comments", "pri_comments", "reference", "ref_id", "pub_lab", "pa_race", "date added"}
display(mmc3[[c for c in mmc3.columns if c not in hidden]].head(4))

,mut_id,g_name,mut_type,mut_effect,d_id,location,e_i_numb,aa_numb,nuc_numb,ntchange,aa_first,aa_last,uinhibitor
0,1,F8,Point,Frameshift,5UTR,UTR,5UTR,0,-860,A>G,NULL,NULL,No
1,2,F8,Point,Frameshift,5UTR,UTR,5UTR,0,-257,T>G,NULL,NULL,NULL
2,3,F8,Point,Frameshift,5UTR,UTR,5UTR,0,-219,C>T,NULL,NULL,NULL
3,4,F8,Point,Frameshift,5UTR,UTR,5UTR,0,-112,G>A,NULL,NULL,No


,mut_id,all_id,subgroups,clotting,discrep,ratio,assay,antigen,act/ant,type,mutations,bleed_tool,bleed_score,cli_phe,Inhibitors
0,1,1,,3,NULL,NULL,,NULL,NA,NULL,Natural,NULL,NULL,Moderate,No
1,2,2,,22,NULL,NULL,,NULL,NA,NULL,Natural,NULL,NULL,Mild,
2,2,3,,29,NULL,NULL,,NULL,NA,NULL,Natural,NULL,NULL,Mild,
3,2,4,,28,NULL,NULL,,NULL,NA,NULL,Natural,NULL,NULL,Mild,


## 3. Original parameter dictionary

One row for every original column of both tables. The columns of the dictionary are built as follows:

| Column | Source |
|---|---|
| **Interpretation** | The project's audit notes (`data_audit.COLUMN_ROLES`), inferred from names and values. *Not verified against the source article's documentation.* |
| **Type / % missing** | Computed in Section 2.2. |
| **Intended use** | The role assigned during the audit. |
| **Cleaned column(s)** | Where the column ends up in `clean_variants` / `clean_reports`. This mapping is checked against the actual function output below. |
| **Final decision** | Derived from the implementation's feature blocks and leakage list. |

In [8]:
# Raw column -> cleaned column(s), as written in pp.clean_variants / pp.clean_reports.
RAW_TO_CLEAN = {
    ("MMC2", "mut_id"): ["mut_id"], ("MMC2", "g_name"): ["gene"], ("MMC2", "mut_type"): ["mut_type"],
    ("MMC2", "mut_effect"): ["mut_effect"], ("MMC2", "d_id"): ["domain"], ("MMC2", "location"): ["location"],
    ("MMC2", "e_i_numb"): ["exon_intron"], ("MMC2", "aa_numb"): ["aa_position"],
    ("MMC2", "nuc_numb"): ["cdna_position"], ("MMC2", "n_bp"): ["n_bp"], ("MMC2", "ntchange"): ["nt_change"],
    ("MMC2", "codon_first"): ["codon_ref"], ("MMC2", "codon_last"): ["codon_alt"],
    ("MMC2", "aa_first"): ["aa_ref"], ("MMC2", "aa_last"): ["aa_alt"], ("MMC2", "mut_syn"): ["hgvs_cdna"],
    ("MMC2", "aa_syn"): ["hgvs_protein"], ("MMC2", "ucomments"): ["annotation"],
    ("MMC3", "all_id"): ["all_id"], ("MMC3", "mut_id"): ["mut_id"], ("MMC3", "Inhibitors"): ["label_status", "y"],
    ("MMC3", "clotting"): ["fviii_activity", "fviii_activity_kind"],
    ("MMC3", "antigen"): ["fviii_antigen", "fviii_antigen_kind"], ("MMC3", "cli_phe"): ["severity"],
    ("MMC3", "type"): ["crm_type"], ("MMC3", "pa_race"): ["patient_origin"],
    ("MMC3", "reference"): ["meta_reference", "is_gnomad"], ("MMC3", "ref_id"): ["meta_ref_id"],
    ("MMC3", "pub_lab"): ["meta_pub_lab"],
}
# Raw columns read by a cleaning rule without being carried over.
RULE_ONLY = {("MMC3", "mutations"): "used to detect shifted rows (find_misaligned)",
             ("MMC3", "bleed_tool"): "used to detect shifted rows (find_misaligned)",
             ("MMC2", "g_id"): "duplicate of g_name; gene filter applied via g_name"}

variants_clean = pp.clean_variants(mmc2)
reports_clean = pp.clean_reports(mmc3)
mapped = {c for cols in RAW_TO_CLEAN.values() for c in cols}
produced = set(variants_clean.columns) | set(reports_clean.columns)
assert mapped <= produced, mapped - produced
assert produced - mapped <= {"is_shifted"}, produced - mapped  # is_shifted is a flag, not a source column
print("Mapping verified: every cleaned column is accounted for (plus the derived flag 'is_shifted').")

Mapping verified: every cleaned column is accounted for (plus the derived flag 'is_shifted').


In [9]:
SOURCE = set(features.SOURCE_COLUMNS)


def final_decision(dataset, column):
    if column in FORBIDDEN_PREDICTORS:
        forbidden = " (on the leakage list)"
    else:
        forbidden = ""
    cleaned = RAW_TO_CLEAN.get((dataset, column))
    if cleaned is None:
        if (dataset, column) in RULE_ONLY:
            return "Not carried over; " + RULE_ONLY[(dataset, column)] + forbidden
        return "Dropped at cleaning (not carried into the cleaned tables)" + forbidden
    if TARGET in cleaned:
        return "TARGET (y); unknown values kept as missing, never negative"
    if column == "reference":
        return "Used to remove gnomAD rows; kept only as metadata, excluded from predictors" + forbidden
    used = [c for c in cleaned if c in SOURCE]
    if used:
        block = "genomic" if used[0] in GENOMIC_COLUMNS else "clinical"
        return f"Selected: {block} predictor source -> {', '.join(cleaned)}"
    if any(c in GENOMIC_COLUMNS + CLINICAL_COLUMNS for c in cleaned):
        return "Carried into fused data as a candidate, not used by the feature pipeline"
    if any(c in ("mut_id", "all_id") for c in cleaned):
        return "Identifier / merge key; never a predictor"
    if any(c in ("hgvs_cdna", "hgvs_protein") for c in cleaned):
        return "Used only to build variant_key (grouping); excluded from predictors" + forbidden
    if column == "g_name":
        return "Filter: keep F8 (haemophilia A) variants only"
    return "Kept as metadata only; excluded from predictors" + forbidden


rows = []
for dataset, table in (("MMC2", "mmc2"), ("MMC3", "mmc3")):
    for column in raw[dataset].columns:
        role, note = data_audit.COLUMN_ROLES[table].get(column, ("unlisted", "not documented"))
        rows.append({
            "column": column, "dataset": dataset,
            "interpretation (inferred, unverified)": note,
            "type": prof.loc[(prof["dataset"] == dataset)].loc[column, "inferred_type"],
            "% missing": prof.loc[(prof["dataset"] == dataset)].loc[column, "pct_missing"],
            "intended use (audit role)": role,
            "cleaned column(s)": ", ".join(RAW_TO_CLEAN.get((dataset, column), [])) or "-",
            "final decision": final_decision(dataset, column),
        })
dictionary = pd.DataFrame(rows)
with pd.option_context("display.max_rows", 80, "display.max_colwidth", 70):
    display(dictionary)

,column,dataset,"interpretation (inferred, unverified)",type,% missing,intended use (audit role),cleaned column(s),final decision
0,mut_id,MMC2,Variant id; unique in MMC2; join and grouping key,numeric,0.00,group_key,mut_id,Identifier / merge key; never a predictor
1,mut_type,MMC2,Point/Deletion/Duplication/...; trailing-space variants,categorical,0.32,genomic,mut_type,Selected: genomic predictor source -> mut_type
2,mut_effect,MMC2,Missense/Frameshift/Nonsense/Splice/Large deletion ...,categorical,0.89,genomic,mut_effect,Selected: genomic predictor source -> mut_effect
3,g_id,MMC2,"g08 (F8) vs g09 (F9); F9 rows are haemophilia B, exclude",categorical,0.00,filter,-,Not carried over; duplicate of g_name; gene filter applied via g_name
4,g_name,MMC2,Duplicate of g_id,categorical,0.00,filter,gene,Filter: keep F8 (haemophilia A) variants only
5,d_id,MMC2,"FVIII protein domain (A1, A2, B, C2, ...)",categorical,0.43,genomic,domain,Selected: genomic predictor source -> domain
6,location,MMC2,Exon/Intron/UTR/Promoter,categorical,0.45,genomic,location,Selected: genomic predictor source -> location
7,e_i_numb,MMC2,Exon or intron number; mixed with text,mixed_numeric_text,1.21,genomic,exon_intron,Selected: genomic predictor source -> exon_intron
8,locnumb,MMC2,Near-duplicate of e_i_numb,mixed_numeric_text,1.00,redundant,-,Dropped at cleaning (not carried into the cleaned tables)
9,aa_numb_old,MMC2,Legacy amino-acid numbering; superseded by aa_numb,numeric,34.97,redundant,-,Dropped at cleaning (not carried into the cleaned tables)


The dictionary agrees with the data dictionary produced in Phase A (`results/audit/column_profile.csv`).
The cell below checks the missingness figures against that file.

In [10]:
saved_profile = pd.read_csv(RESULTS / "audit" / "column_profile.csv")
saved_profile["dataset"] = saved_profile["table"].str.upper()
merged = dictionary.merge(saved_profile[["dataset", "column", "pct_missing"]], on=["dataset", "column"])
mismatch = (merged["% missing"] - merged["pct_missing"]).abs() > 0.01
print(f"{len(merged)} columns compared with the Phase A profile; missingness mismatches: {int(mismatch.sum())}")

57 columns compared with the Phase A profile; missingness mismatches: 0


## 4. Existing cleaning procedure

Cleaning happens in two stages:

1. **Cell and table cleaning** (`src/preprocessing.py`). This types each column, maps spelling
   variants and sentinels, parses FVIII values and maps the outcome labels. It produces one cleaned
   variant table and one cleaned report table.
2. **Record exclusions** (`fusion.apply_exclusions`). This removes report rows in a fixed order and
   counts every step.

Both stages are shown below with the code that implements them.

### 4.1 Typing, sentinels and spelling variants

In [11]:
show_source(pp.clean_variants)
show_source(pp.clean_reports)

```python
def clean_variants(mmc2: pd.DataFrame) -> pd.DataFrame:
    """Typed variant table, one row per ``mut_id``.

    Mutation-level patient summaries (``uinhibitor``, ``useverity``,
    ``uclotting`` ...), the stale ``Count_mut_id`` and curation dates are not
    carried over.
    """
    if not mmc2["mut_id"].is_unique:
        raise ValueError("MMC2 mut_id is not unique; cannot build a variant table")
    aa_position = to_number(mmc2["aa_numb"])
    return pd.DataFrame(
        {
            "mut_id": mmc2["mut_id"].str.strip(),
            "gene": clean_category(mmc2["g_name"]),
            "mut_type": clean_category(mmc2["mut_type"], MUT_TYPE_MAP),
            "mut_effect": clean_category(mmc2["mut_effect"], MUT_EFFECT_MAP),
            "domain": clean_category(mmc2["d_id"], DOMAIN_MAP),
            "location": clean_category(mmc2["location"], LOCATION_MAP),
            "exon_intron": clean_category(mmc2["e_i_numb"]),
            # 0 is used for variants with no amino-acid position (UTR, intron).
            "aa_position": aa_position.where(aa_position != 0),
            "cdna_position": to_number(mmc2["nuc_numb"]),
            "n_bp": to_number(mmc2["n_bp"]),
            "nt_change": clean_category(mmc2["ntchange"]),
            "codon_ref": clean_category(mmc2["codon_first"]),
            "codon_alt": clean_category(mmc2["codon_last"]),
            "aa_ref": clean_category(mmc2["aa_first"]),
            "aa_alt": clean_category(mmc2["aa_last"]),
            "hgvs_cdna": hgvs_key(mmc2["mut_syn"]),
            "hgvs_protein": protein_key(mmc2["aa_syn"]),
            "annotation": clean_category(mmc2["ucomments"]),
        }
    )
```

```python
def clean_reports(mmc3: pd.DataFrame) -> pd.DataFrame:
    """Typed allele-report table, one row per ``all_id``, with row flags."""
    if not mmc3["all_id"].is_unique:
        raise ValueError("MMC3 all_id is not unique")
    labels = map_inhibitor_labels(mmc3["Inhibitors"])
    activity = pd.DataFrame(
        mmc3["clotting"].map(parse_fviii).tolist(),
        columns=["fviii_activity", "fviii_activity_kind"],
        index=mmc3.index,
    )
    antigen = pd.DataFrame(
        mmc3["antigen"].map(parse_fviii).tolist(),
        columns=["fviii_antigen", "fviii_antigen_kind"],
        index=mmc3.index,
    )
    shifted = find_misaligned(mmc3)
    severity = clean_category(mmc3["cli_phe"]).map(
        lambda v: SEVERITY_MAP.get(v, "other") if isinstance(v, str) else np.nan
    )
    return pd.DataFrame(
        {
            "all_id": mmc3["all_id"].str.strip(),
            "mut_id": mmc3["mut_id"].str.strip(),
            "label_status": labels["label_status"],
            "y": labels["y"],
            "fviii_activity": activity["fviii_activity"].astype(float),
            "fviii_activity_kind": activity["fviii_activity_kind"],
            "fviii_antigen": antigen["fviii_antigen"].astype(float),
            "fviii_antigen_kind": antigen["fviii_antigen_kind"],
            # Severity from shifted rows is unreliable and left missing.
            "severity": severity.where(~shifted),
            "crm_type": clean_category(mmc3["type"]).where(lambda s: s.isin({"i", "ii"})),
            "patient_origin": clean_category(mmc3["pa_race"]),
            # Provenance: kept for audit and stratified error analysis only.
            "meta_reference": clean_text(mmc3["reference"]),
            "meta_ref_id": clean_text(mmc3["ref_id"]),
            "meta_pub_lab": clean_text(mmc3["pub_lab"]),
            "is_gnomad": mmc3["reference"].map(normalise_token).eq(GNOMAD_REFERENCE),
            "is_shifted": shifted,
        }
    )
```

In [12]:
maps = {"mut_type": pp.MUT_TYPE_MAP, "mut_effect": pp.MUT_EFFECT_MAP, "location": pp.LOCATION_MAP,
        "domain (d_id)": pp.DOMAIN_MAP}
for name, mapping in maps.items():
    print(f"{name:14s}", mapping)
before_after = pd.DataFrame({
    "raw distinct values": [mmc2["mut_type"].nunique(), mmc2["mut_effect"].nunique(), mmc2["d_id"].nunique(),
                            mmc2["location"].nunique(), mmc3["cli_phe"].nunique()],
    "cleaned distinct values": [variants_clean["mut_type"].nunique(), variants_clean["mut_effect"].nunique(),
                                variants_clean["domain"].nunique(), variants_clean["location"].nunique(),
                                reports_clean["severity"].nunique()],
}, index=["mut_type", "mut_effect", "d_id -> domain", "location", "cli_phe -> severity"])
display(before_after)
print("aa_numb == 0 (no amino-acid position, e.g. UTR/intron) set to missing:",
      int((pp.to_number(mmc2["aa_numb"]) == 0).sum()), "variants")
print("CRM 'type' kept only when 'i' or 'ii':", reports_clean["crm_type"].value_counts().to_dict())

mut_type       {'duptication': 'duplication'}
mut_effect     {'inframe': 'in-frame', 'splicing': 'splice'}
location       {'promote': 'promoter', 'intron/': 'intron', 'exon 5': 'exon', '5utr -': 'utr'}
domain (d_id)  {'5-utr': '5utr', '3-utr': '3utr', 'a1-a2': 'multiple domains', 'a1-a1-a2': 'multiple domains', 'a3-c1': 'multiple domains', 'c1-c2': 'multiple domains', 'undefined': None}


,raw distinct values,cleaned distinct values
mut_type,13,6
mut_effect,25,9
d_id -> domain,29,14
location,12,4
cli_phe -> severity,29,7


aa_numb == 0 (no amino-acid position, e.g. UTR/intron) set to missing: 625 variants
CRM 'type' kept only when 'i' or 'ii': {'ii': 242, 'i': 216}


Trailing spaces, capitalisation and a few typos (`Duptication`, `Svere`, `Moderare`) split the same
category into several raw values. The cleaning maps them to one value each. Values that the mapping
does not recognise are kept, not discarded.

### 4.2 Inhibitor outcome labels (unknown is never negative)

In [13]:
show_source(labels.label_status)
mapping = labels.map_inhibitor_labels(mmc3["Inhibitors"])
table = (pd.DataFrame({"raw value": mmc3["Inhibitors"].map(repr), "status": mapping["label_status"],
                       "y": mapping["y"].astype("string").fillna("missing (unknown)")})
         .value_counts().rename("reports").reset_index())
display(table)
unknown = mapping["label_status"].isin([labels.UNKNOWN_NOT_REPORTED, labels.UNKNOWN_BLANK, labels.INVALID_VALUE])
assert mapping.loc[unknown, "y"].isna().all(), "an unknown label was given a class"
print(f"Labelled: {int(mapping['y'].notna().sum()):,} (positive {int((mapping['y'] == 1).sum())}, "
      f"negative {int((mapping['y'] == 0).sum()):,}); unknown/invalid kept as missing: {int(unknown.sum()):,}")

```python
def label_status(value: object) -> str:
    """Map one raw ``Inhibitors`` cell to a label status.

    Values outside the known vocabulary (for example "Mild" in rows whose
    columns are shifted) are ``invalid_value``, not negative.
    """
    return _RAW_TO_STATUS.get(normalise_token(value), INVALID_VALUE)
```

,raw value,status,y,reports
0,'No',confirmed_negative,0,3923
1,'Not reported',unknown_not_reported,missing (unknown),2090
2,'',unknown_blank,missing (unknown),1730
3,'Not',unknown_not_reported,missing (unknown),1276
4,'Yes',confirmed_positive,1,835
5,'No ',confirmed_negative,0,207
6,'Mild',invalid_value,missing (unknown),1
7,'Severe',invalid_value,missing (unknown),1
8,'yes',confirmed_positive,1,1


Labelled: 4,966 (positive 836, negative 4,130); unknown/invalid kept as missing: 5,098


`Not reported`, the truncated `Not`, blank cells and invalid values (severity words in shifted rows)
are **unknown**. They are never converted to "no inhibitor". Converting them would invent negatives and
bias the model towards predicting no inhibitor.

### 4.3 Malformed (shifted-column) rows

In [14]:
show_source(pp.find_misaligned)
shifted = pp.find_misaligned(mmc3)
display(mmc3.loc[shifted, ["all_id", "Inhibitors", "mutations", "bleed_tool", "cli_phe"]])
print("Shifted rows:", int(shifted.sum()))

```python
def find_misaligned(mmc3: pd.DataFrame) -> pd.Series:
    """MMC3 rows whose trailing columns are shifted one place right."""
    status = map_inhibitor_labels(mmc3["Inhibitors"])["label_status"]
    mutations = mmc3["mutations"].map(normalise_token)
    return (
        (status == "invalid_value")
        | ~mutations.isin({"natural", ""})
        | mmc3["bleed_tool"].map(normalise_token).eq("natural")
    )
```

,all_id,Inhibitors,mutations,bleed_tool,cli_phe
971,973,Mild,01/11/2005,Natural,NULL
2305,2309,Severe,01/12/1996,Natural,NULL


Shifted rows: 2


In these rows every field from some point onwards has moved one column to the right. As a result, the
`Inhibitors` cell holds a severity word and `mutations` holds a date. Their outcome and severity cannot
be trusted, so the rows are removed.

### 4.4 Population-only gnomAD records

In [15]:
gnomad = reports_clean["is_gnomad"]
print("Rows whose reference is", repr(pp.GNOMAD_REFERENCE), ":", int(gnomad.sum()))
display(reports_clean.loc[gnomad, "label_status"].value_counts().to_frame("gnomAD rows by label status"))

Rows whose reference is 'gnomad v2.1.1' : 1289


,gnomAD rows by label status
label_status,
unknown_not_reported,1289


These rows describe variant frequencies in the gnomAD population database, not haemophilia patients.
None has a usable inhibitor outcome, and they are removed before anything else is counted.

### 4.5 Identifier problems: unmatched `mut_id`, F9 variants, stale `Count_mut_id`

In [16]:
unmatched = sorted(set(reports_clean["mut_id"]) - set(variants_clean["mut_id"]))
print("MMC3 mut_id values with no MMC2 record:", unmatched,
      "| reports affected:", int(reports_clean["mut_id"].isin(unmatched).sum()))
print("Gene values in MMC2:", variants_clean["gene"].value_counts(dropna=False).to_dict())
f9 = set(variants_clean.loc[variants_clean["gene"] != pp.F8_GENE, "mut_id"])
print("Reports pointing to non-F8 variants:", int(reports_clean["mut_id"].isin(f9).sum()))

recount = mmc3["mut_id"].value_counts()
stated = pp.to_number(mmc2.set_index("mut_id")["Count_mut_id"])
compare = pd.DataFrame({"Count_mut_id": stated, "MMC3 rows": recount}).fillna(0)
print(f"Count_mut_id disagrees with the actual MMC3 row count for {int((compare.iloc[:, 0] != compare.iloc[:, 1]).sum()):,}"
      f" of {len(compare):,} mut_id values")

MMC3 mut_id values with no MMC2 record: ['1255'] | reports affected: 1
Gene values in MMC2: {'f8': 6205, 'f9': 6}
Reports pointing to non-F8 variants: 6
Count_mut_id disagrees with the actual MMC3 row count for 2,779 of 6,212 mut_id values


- The **unmatched** report has no variant to take genomic fields from, so it cannot be merged and is removed.
- **F9** variants belong to haemophilia B and are outside the project's scope.
- **`Count_mut_id`** is not used at all. It disagrees with the real report counts, and a report count
  is a property of the literature, not of the variant.

### 4.6 Ordered exclusions and the resulting dimensions

In [17]:
show_source(fusion.apply_exclusions)
key = fusion.build_variant_key(variants_clean, CONFIG.link_on_hgvs_cdna, CONFIG.link_on_hgvs_protein)
variants_keyed = variants_clean.assign(**{GROUP_KEY: variants_clean["mut_id"].map(key)})
kept, flow = fusion.apply_exclusions(reports_clean, variants_keyed)
display(flow[["step", "reason", "removed", "removed_positives", "removed_negatives", "rows", "positives", "negatives"]])
saved_flow = pd.read_csv(RESULTS / "integration" / "record_flow.csv")
pd.testing.assert_frame_equal(flow.reset_index(drop=True), saved_flow, check_dtype=False)
print("Re-running the exclusions in memory reproduces the saved results/integration/record_flow.csv exactly.")

```python
def apply_exclusions(reports: pd.DataFrame, variants: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Apply the documented exclusions in order; return kept rows and the flow table."""
    known_ids = set(variants["mut_id"])
    f8_ids = set(variants.loc[variants["gene"] == F8_GENE, "mut_id"])
    steps: list[tuple[str, str, Callable[[pd.DataFrame], pd.Series]]] = [
        ("shifted_rows", "Columns shifted one place; outcome cell holds a severity word",
         lambda f: f["is_shifted"]),
        ("gnomad_population_rows", "gnomAD population variants, not patients",
         lambda f: f["is_gnomad"]),
        ("mut_id_not_in_mmc2", "No variant record to join genomic features from",
         lambda f: ~f["mut_id"].isin(known_ids)),
        ("non_f8_variant", "F9 (haemophilia B) variant",
         lambda f: ~f["mut_id"].isin(f8_ids)),
        ("unlabelled_outcome", "Inhibitors not reported / blank: unknown, not negative",
         lambda f: f[TARGET].isna()),
    ]
    flow = [
        {
            "step": "raw_mmc3",
            "reason": "All allele reports",
            "removed": 0,
            "removed_positives": 0,
            "removed_negatives": 0,
            **_counts(reports),
        }
    ]
    kept = reports
    for name, reason, rule in steps:
        drop = rule(kept)
        removed = kept[drop]
        kept = kept[~drop]
        flow.append(
            {
                "step": name,
                "reason": reason,
                "removed": len(removed),
                "removed_positives": int((removed[TARGET] == 1).sum()),
                "removed_negatives": int((removed[TARGET] == 0).sum()),
                **_counts(kept),
            }
        )
    return kept.copy(), pd.DataFrame(flow)
```

,step,reason,removed,removed_positives,removed_negatives,rows,positives,negatives
0,raw_mmc3,All allele reports,0,0,0,10064,836,4130
1,shifted_rows,Columns shifted one place; outcome cell holds a severity word,2,0,0,10062,836,4130
2,gnomad_population_rows,"gnomAD population variants, not patients",1289,0,0,8773,836,4130
3,mut_id_not_in_mmc2,No variant record to join genomic features from,1,0,1,8772,836,4129
4,non_f8_variant,F9 (haemophilia B) variant,6,0,3,8766,836,4126
5,unlabelled_outcome,"Inhibitors not reported / blank: unknown, not negative",3804,0,0,4962,836,4126


Re-running the exclusions in memory reproduces the saved results/integration/record_flow.csv exactly.

The order matters. Structural problems (shifted rows, gnomAD rows, missing or non-F8 variants) are
removed first, so the last step only removes reports whose outcome is genuinely unknown. 4 labelled
reports are removed before that step: the unmatched one and 3 F9 negatives.

### 4.7 Conflicting outcomes within a variant

In [18]:
groups_saved = pd.read_csv(PROCESSED / "variant_groups.csv")
display(groups_saved["outcome"].value_counts().to_frame("variant groups"))
conflicting = groups_saved[groups_saved["outcome"] == labels.GROUP_CONFLICTING]
print(f"Reports in conflicting groups: {int(conflicting['n_reports'].sum()):,} (kept with their own labels)")

,variant groups
outcome,
all_negative,1905
all_positive,433
conflicting,135


Reports in conflicting groups: 1,400 (kept with their own labels)


Patients carrying the same variant can have different outcomes. These **conflicting** groups are kept,
each report with its own label, because the disagreement is real clinical variability. Removing them would make
the task look easier than it is. The group outcome is used only to stratify the train/validation/test split.

### 4.8 Invalid exon values from spreadsheet date conversion

In [19]:
fused = pd.read_csv(PROCESSED / "fused_reports.csv", dtype={"all_id": str, "mut_id": str, GROUP_KEY: str})
exon_text = fused["exon_intron"].astype("string")
digits = pd.to_numeric(exon_text.where(exon_text.str.fullmatch(r"\d+", na=False)), errors="coerce")
serial = digits > features.MAX_EXON
as_date = pd.to_datetime(digits[serial], unit="D", origin="1899-12-30")
print(f"F8 has {features.MAX_EXON} exons. Reports with a whole-number exon value above that: {int(serial.sum())} "
      f"({digits[serial].nunique()} distinct values, {int(digits[serial].min())}-{int(digits[serial].max())})")
print("Read as spreadsheet date serials they fall in", as_date.min().date(), "to", as_date.max().date())
display(pd.DataFrame({"raw value": digits[serial].astype(int), "as a date": as_date.dt.date})
        .drop_duplicates().head(6))
show_source(features._exon_intron_number)
derived_exon = features.derive_features(fused)["exon_intron_number"]
assert derived_exon[serial].isna().all()
print("All", int(serial.sum()), "date-serial values become missing in the derived exon feature.")

F8 has 26 exons. Reports with a whole-number exon value above that: 69 (46 distinct values, 41306-46023)
Read as spreadsheet date serials they fall in 2013-02-01 to 2026-01-01


,raw value,as a date
432,44507,2021-11-07
734,41609,2013-12-01
3461,44228,2021-02-01
3462,44317,2021-05-01
3463,44348,2021-06-01
3468,44378,2021-07-01


```python
def _exon_intron_number(series: pd.Series) -> pd.Series:
    """Numeric exon/intron number; ranges ("15-22") and UTR labels become NaN.

    F8 has 26 exons and 25 introns. 69 reports carry 5-digit values
    (41306-46023) that match spreadsheet date serials for 2013-2026, apparently
    exon ranges such as "1-2" turned into dates; they are treated as missing.
    """
    text = _text(series)
    number = pd.to_numeric(text.where(text.str.fullmatch(r"\d+", na=False)), errors="coerce")
    return number.where(number.between(1, MAX_EXON))
```

All 69 date-serial values become missing in the derived exon feature.


Values such as 44228 are not exon numbers. Read as spreadsheet date serials they correspond to dates
between February 2013 and January 2026 (cell above). The most likely origin is an exon range such as "1-2" that a spreadsheet
converted into a date, but **that origin is inferred, not verified**.

The implementation does not try to reconstruct the original ranges. It treats these values, like ranges
written as text ("15-22"), as a missing exon number. The raw value stays in the cleaned data for
traceability. This fix was made after the explainability step exposed the outliers (see the README).

## 5. Feature selection and parameter justification

Every original parameter is classified below. Each decision and reason comes from the implementation:
- the feature blocks and the leakage list in `src/fusion.py`;
- the inputs used by `src/features.py`;
- the audit notes.

The evidence cells after the table reproduce the measurements behind the less obvious exclusions.

In [20]:
CATEGORY = {
    ("MMC3", "Inhibitors"): ("Target variable", "Patient-level inhibitor outcome; Yes/No only, unknown kept missing"),
    ("MMC2", "uinhibitor"): ("Redundant or leakage-prone", "Variant-level summary of the patients' inhibitor outcomes: a target proxy"),
    ("MMC2", "useverity"): ("Redundant or leakage-prone", "Variant-level summary of patient data (u* field)"),
    ("MMC2", "uclotting"): ("Redundant or leakage-prone", "Variant-level summary of patient data (u* field)"),
    ("MMC2", "uantigen"): ("Redundant or leakage-prone", "Variant-level summary of patient data (u* field)"),
    ("MMC2", "udiscrep"): ("Redundant or leakage-prone", "Variant-level summary; nearly empty"),
    ("MMC2", "uratio"): ("Redundant or leakage-prone", "Variant-level summary; almost all 0"),
    ("MMC2", "utype"): ("Redundant or leakage-prone", "Variant-level summary; nearly empty"),
    ("MMC2", "Count_mut_id"): ("Excluded or unreliable", "Stale report count (Section 4.5); a literature property, not biology"),
    ("MMC2", "ucomments"): ("Excluded or unreliable", "Curator wording tracks curation batch, not biology (evidence below)"),
    ("MMC2", "mut_syn"): ("Merge keys and variant-group identifiers", "HGVS cDNA name: nearly unique, would be memorised; used to build variant_key"),
    ("MMC2", "aa_syn"): ("Merge keys and variant-group identifiers", "HGVS protein name: used to build variant_key"),
    ("MMC2", "mut_id"): ("Merge keys and variant-group identifiers", "Join key between MMC2 and MMC3"),
    ("MMC3", "mut_id"): ("Merge keys and variant-group identifiers", "Join key to MMC2 (many reports per variant)"),
    ("MMC3", "all_id"): ("Merge keys and variant-group identifiers", "Report identifier; never a feature"),
    ("MMC3", "reference"): ("Metadata", "Source publication; selection-bias proxy (evidence below); also flags gnomAD rows"),
    ("MMC3", "ref_id"): ("Metadata", "Publication id; selection-bias proxy"),
    ("MMC3", "pub_lab"): ("Metadata", "Reporting laboratory country; site proxy"),
    ("MMC3", "pa_race"): ("Excluded or unreliable", "Patient country; mostly one publication per country, so a publication proxy (evidence below)"),
    ("MMC3", "pa_id"): ("Excluded or unreliable", "Patient code within a publication; identifier"),
    ("MMC3", "comments"): ("Excluded or unreliable", "Free text; can mention inhibitors"),
    ("MMC3", "pri_comments"): ("Excluded or unreliable", "Free text; can mention inhibitors"),
    ("MMC2", "g_name"): ("Metadata", "Gene; used only to keep F8 variants"),
    ("MMC2", "g_id"): ("Metadata", "Duplicate of g_name"),
}
GENOMIC_RAW = {"mut_type", "mut_effect", "d_id", "location", "e_i_numb", "aa_numb", "nuc_numb", "n_bp",
               "ntchange", "codon_first", "aa_first", "aa_last"}
CLINICAL_RAW = {"clotting", "antigen", "cli_phe", "type"}

selection_rows = []
for _, r in dictionary.iterrows():
    key = (r["dataset"], r["column"])
    if key in CATEGORY:
        category, reason = CATEGORY[key]
    elif r["dataset"] == "MMC2" and r["column"] in GENOMIC_RAW:
        category, reason = "Genomic predictors", "Describes the variant; source of genomic features (Section 7)"
    elif r["dataset"] == "MMC3" and r["column"] in CLINICAL_RAW:
        category, reason = "Clinical predictors", "Describes the patient's disease; source of clinical features"
    elif key == ("MMC2", "codon_last"):
        category, reason = "Redundant or leakage-prone", "Variant codon is near-determined by codon_first + ntchange; carried in fused data but unused"
    elif r["dataset"] == "MMC2" and r["column"] in {"locnumb", "aa_numb_old", "codon_change", "aa_change"}:
        category, reason = "Redundant or leakage-prone", "Duplicates or combines information already in selected columns"
    else:
        category, reason = "Excluded or unreliable", r["interpretation (inferred, unverified)"]
    decision = "Used" if category in ("Target variable", "Genomic predictors", "Clinical predictors") else (
        "Kept, not a predictor" if category in ("Merge keys and variant-group identifiers", "Metadata") else "Excluded")
    selection_rows.append({"original parameter": r["column"], "dataset": r["dataset"], "decision": decision,
                           "reason": reason, "final role": category})
selection = pd.DataFrame(selection_rows)
with pd.option_context("display.max_rows", 80, "display.max_colwidth", 95):
    display(selection.sort_values(["final role", "dataset", "original parameter"]).reset_index(drop=True))
display(selection["final role"].value_counts().to_frame("parameters"))

,original parameter,dataset,decision,reason,final role
0,antigen,MMC3,Used,Describes the patient's disease; source of clinical features,Clinical predictors
1,cli_phe,MMC3,Used,Describes the patient's disease; source of clinical features,Clinical predictors
2,clotting,MMC3,Used,Describes the patient's disease; source of clinical features,Clinical predictors
3,type,MMC3,Used,Describes the patient's disease; source of clinical features,Clinical predictors
4,Count_mut_id,MMC2,Excluded,"Stale report count (Section 4.5); a literature property, not biology",Excluded or unreliable
5,CpG,MMC2,Excluded,Effectively empty,Excluded or unreliable
6,Date added to HADB,MMC2,Excluded,Curation date; mixed formats,Excluded or unreliable
7,mutations,MMC2,Excluded,Always 'Natural',Excluded or unreliable
8,ucomments,MMC2,Excluded,"Curator wording tracks curation batch, not biology (evidence below)",Excluded or unreliable
9,act/ant,MMC3,Excluded,Activity/antigen ratio; ~96% missing,Excluded or unreliable


,parameters
final role,
Excluded or unreliable,18
Redundant or leakage-prone,12
Genomic predictors,12
Metadata,5
Merge keys and variant-group identifiers,5
Clinical predictors,4
Target variable,1


In [21]:
# Consistency with the implementation: every raw column marked "Used" as a predictor feeds a column
# that src/features.py actually reads, and nothing marked "Excluded" does.
used_clean = {c for _, r in selection.iterrows() if r["final role"] in ("Genomic predictors", "Clinical predictors")
              for c in RAW_TO_CLEAN[(r["dataset"], r["original parameter"])]}
assert used_clean <= SOURCE | {"fviii_activity_kind", "fviii_antigen_kind"}, used_clean - SOURCE
not_used = {c for _, r in selection.iterrows() if r["decision"] != "Used"
            for c in RAW_TO_CLEAN.get((r["dataset"], r["original parameter"]), [])}
assert not (not_used & SOURCE), not_used & SOURCE
print("Selection table is consistent with src/features.SOURCE_COLUMNS.")

Selection table is consistent with src/features.SOURCE_COLUMNS.


**Engineered features** (Section 7) are not original parameters. They are computed from the selected
genomic and clinical columns, and they are what the models receive after encoding.

### 5.1 Evidence behind the key exclusions

**`Inhibitors` as the outcome** is the only patient-level outcome field (Section 4.2).

**`uinhibitor` is a target proxy.** It summarises the outcome of the patients carrying a variant, so it
almost always agrees with the outcome the model is asked to predict. The comparison below is the one
saved by the Phase A audit:

In [22]:
display(pd.read_csv(RESULTS / "audit" / "uinhibitor_vs_group_outcome.csv"))

,uinhibitor,all_negative,all_positive,conflicting
0,(blank),1,2,0
1,(mut_id not in mmc2),1,0,0
2,no,1548,3,0
3,not reported,6,1,1
4,NaN,466,44,118
5,yes,1,446,5


**Publication and country fields carry selection bias.** Some publications report only patients with
inhibitors, and patient country mostly identifies one publication. The cell below measures both on the
final dataset:

In [23]:
by_pub = fused.groupby("meta_ref_id")["y"].agg(["size", "mean"])
pos_only = by_pub[by_pub["mean"] == 1]
neg_only = by_pub[by_pub["mean"] == 0]
print(f"Publications: {len(by_pub)}; only-positive publications: {len(pos_only)} supplying "
      f"{int(pos_only['size'].sum())} of {int(fused['y'].sum())} positives; only-negative: {len(neg_only)}")
origin = fused.dropna(subset=["patient_origin"])
top_share = origin.groupby("patient_origin")["meta_ref_id"].agg(lambda s: s.value_counts(normalize=True).iloc[0])
weight = origin["patient_origin"].value_counts()
print(f"Patient country: on average {100 * (top_share * weight).sum() / weight.sum():.0f}% of a country's "
      f"reports come from that country's largest publication")

Publications: 322; only-positive publications: 56 supplying 247 of 836 positives; only-negative: 140
Patient country: on average 66% of a country's reports come from that country's largest publication


**Curator comments (`ucomments`) follow the curation batch.** Near-identical wordings have very different positive rates:

In [24]:
ann = fused.assign(annotation=fused["annotation"].fillna("(missing)"))
display(ann.groupby("annotation")["y"].agg(reports="size", positive_rate="mean")
        .query("reports >= 10").sort_values("reports", ascending=False).round(3))

,reports,positive_rate
annotation,,
details in multiple view,3112,0.130
heavy chain affected,860,0.141
light chain affected,420,0.238
(missing),258,0.380
large structural change affecting multiple domains,31,0.613
heavy chain,22,0.682
light chain,19,0.368
large structural change affecting a single domain,16,0.562
female,12,0.167


"heavy chain" and "heavy chain affected" describe the same thing but have very different positive
rates. "details in Multiple view" marks variants with many reports. The comment therefore carries
information about how a variant was curated, not about its biology. Chain information is already in
`domain`.

**Redundant variables** are dropped in the `combined_minus_redundant` ablation set:

In [25]:
rho = fused[["aa_position", "cdna_position"]].corr(method="spearman").iloc[0, 1]
print(f"Spearman correlation of aa_position and cdna_position: {rho:.3f}")
print("Redundant set (dropped in the ablation):", features.REDUNDANT)
display(pd.read_csv(RESULTS / "audit" / "severity_vs_fviii.csv").head(8))

Spearman correlation of aa_position and cdna_position: 0.996
Redundant set (dropped in the ablation): ('aa_position', 'log_fviii_activity', 'fviii_activity_kind')


,cli_phe,mild(>5),moderate(1-5),severe(<1)
0,(blank),51,0,0
1,mild,1159,216,17
2,mild/moderate,27,17,0
3,mild/severe,0,1,0
4,moderate,145,576,9
5,moderate/severe,3,5,3
6,not reported,10,1,5
7,severe,438,151,2117


Clinical severity is defined from residual FVIII activity, so the two clinical variables overlap. Both
are kept in the main (`combined`) feature set. The ablation measures what removing the redundant ones
costs.

**Leakage guard.** `fusion.check_no_leakage` raises an error if any feature set contains a forbidden
column:

In [26]:
show_source(fusion.check_no_leakage)
print("Forbidden predictors:", sorted(FORBIDDEN_PREDICTORS))
for name, spec in features.FEATURE_SETS.items():
    fusion.check_no_leakage(spec["numeric"] + spec["categorical"])
print("All", len(features.FEATURE_SETS), "feature sets pass the leakage check.")

```python
def check_no_leakage(feature_columns: list[str] | tuple[str, ...]) -> None:
    """Raise if any candidate predictor is a target, proxy or identifier."""
    bad = sorted(set(feature_columns) & FORBIDDEN_PREDICTORS)
    if bad:
        raise ValueError(f"Forbidden predictor columns: {bad}")
```

Forbidden predictors: ['Count_mut_id', 'Date added to HADB', 'Inhibitors', 'aa_syn', 'all_id', 'annotation', 'comments', 'date added', 'hgvs_cdna', 'hgvs_protein', 'label_status', 'meta_pub_lab', 'meta_ref_id', 'meta_reference', 'mut_id', 'mut_syn', 'pa_id', 'pa_race', 'patient_origin', 'pri_comments', 'pub_lab', 'ref_id', 'reference', 'uantigen', 'uclotting', 'ucomments', 'udiscrep', 'uinhibitor', 'uratio', 'useverity', 'utype', 'variant_key', 'y']
All 5 feature sets pass the leakage check.


## 6. MMC2–MMC3 integration

### 6.1 Relationship and merge key

MMC2 holds one row per variant and MMC3 one row per report, so each report is joined to **exactly one**
variant on `mut_id` (many-to-one). The merge is a pandas left join with `validate="many_to_one"`. The
code also raises an error if the row count changes, a report finds no variant, or a report id repeats.

In [27]:
show_source(fusion.merge_reports_variants)
reports_per_variant = kept["mut_id"].value_counts()
display(reports_per_variant.describe().to_frame("labelled reports per mut_id").T)
merged_now = fusion.merge_reports_variants(kept.drop(columns=["is_gnomad", "is_shifted"]), variants_keyed)
display(pd.DataFrame({"rows before merge": [len(kept)], "rows after merge": [len(merged_now)],
                      "rows created by the merge": [len(merged_now) - len(kept)],
                      "reports without a variant": [int(merged_now["gene"].isna().sum())]}))

```python
def merge_reports_variants(reports: pd.DataFrame, variants: pd.DataFrame) -> pd.DataFrame:
    """Join variant fields onto reports, many-to-one on ``mut_id``.

    Raises if the join would duplicate or lose a report, or if a report has
    no matching variant.
    """
    merged = reports.merge(
        variants, on="mut_id", how="left", validate="many_to_one", indicator=True
    )
    if len(merged) != len(reports):
        raise ValueError(f"Merge changed row count: {len(reports)} -> {len(merged)}")
    unmatched = int((merged["_merge"] != "both").sum())
    if unmatched:
        raise ValueError(f"{unmatched} reports have no variant record after exclusions")
    if not merged[ROW_ID].is_unique:
        raise ValueError("Merge produced duplicate report ids")
    return merged.drop(columns="_merge")
```

,count,mean,std,min,25%,50%,75%,max
labelled reports per mut_id,2639.0,1.88,4.467,1.0,1.0,1.0,1.0,104.0


,rows before merge,rows after merge,rows created by the merge,reports without a variant
0,4962,4962,0,0


**What the guard prevents.** If a `mut_id` appeared twice on the variant side, the join would silently
copy reports. The demonstration below duplicates one variant and shows that the implementation refuses
the merge:

In [28]:
broken = pd.concat([variants_keyed, variants_keyed.iloc[[0]]])
try:
    fusion.merge_reports_variants(kept.drop(columns=["is_gnomad", "is_shifted"]), broken)
    print("merge accepted (unexpected)")
except pd.errors.MergeError as exc:
    print("Merge refused:", str(exc).splitlines()[0])

Merge refused: Merge keys are not unique in right dataset; not a many-to-one merge


### 6.2 Different `mut_id`s naming the same variant

`mut_id` alone does not identify a biological variant uniquely. Some HGVS names appear under several
`mut_id`s. If those were split across train and test, the test set would contain variants the model had
already seen.

`fusion.build_variant_key` therefore links `mut_id`s that share an HGVS cDNA name or an HGVS protein
name, transitively (union-find), into one `variant_key`. That key is the unit kept together by the split.

In [29]:
show_source(fusion.build_variant_key)
multi = (variants_keyed.loc[variants_keyed["mut_id"].isin(fused["mut_id"])]
         .dropna(subset=["hgvs_cdna"]).groupby("hgvs_cdna")["mut_id"].nunique())
print(f"HGVS cDNA names used by more than one mut_id in the final data: {int((multi > 1).sum())}")
summary = json.loads((RESULTS / "integration" / "integration_summary.json").read_text())
display(pd.DataFrame(summary["grouping_sensitivity"]).T)
largest = groups_saved.sort_values("n_mut_ids", ascending=False).iloc[0]
print(f"Largest merged group {largest['variant_key']}: {largest['n_mut_ids']} mut_ids, "
      f"{largest['n_reports']} reports, effect '{largest['mut_effect']}'")

```python
def build_variant_key(
    variants: pd.DataFrame, link_on_hgvs_cdna: bool = True, link_on_hgvs_protein: bool = True
) -> pd.Series:
    """Map each ``mut_id`` to a variant key.

    ``mut_id``s are linked when they share a non-missing HGVS cDNA name
    and/or HGVS protein name; links are transitive. A variant with no shared
    name keeps its own key. Missing names never link anything.
    """
    ids = variants["mut_id"].tolist()
    uf = _UnionFind(ids)
    link_columns = [
        column
        for column, enabled in (("hgvs_cdna", link_on_hgvs_cdna), ("hgvs_protein", link_on_hgvs_protein))
        if enabled
    ]
    for column in link_columns:
        named = variants.dropna(subset=[column])
        for members in named.groupby(column)["mut_id"]:
            first, *rest = members[1].tolist()
            for other in rest:
                uf.union(first, other)
    return pd.Series({i: f"v{uf.find(i)}" for i in ids}, name=GROUP_KEY).rename_axis("mut_id")
```

HGVS cDNA names used by more than one mut_id in the final data: 15


,groups,conflicting_groups,largest_group_reports,reports_in_multi_mut_id_groups
mut_id_only,2639,124,104,0
cdna_only,2572,121,104,124
cdna_and_protein,2473,135,104,417


Largest merged group v1860: 43 mut_ids, 64 reports, effect 'large deletion'


The configuration (`configs/data.json`) links on both cDNA and protein names. This is the more cautious
choice: two nucleotide changes with the same protein consequence have near-identical features and are
treated as one unseen variant. Linking on cDNA names only would give 2,572 groups instead of 2,473
(table above).

## 7. Feature engineering already implemented

`features.derive_features` computes new columns **from each row alone**. It uses no labels and no other
rows, so it cannot leak information between partitions.

A `ColumnTransformer`, fitted on the **training partition only**, then imputes, scales and one-hot
encodes the result:
- numeric columns: median imputation, with a missing-value indicator, then scaling;
- categorical columns: a "missing" level, then one-hot encoding, with rare and unseen levels pooled.

In [30]:
show_source(features.derive_features)

```python
def derive_features(frame: pd.DataFrame) -> pd.DataFrame:
    """Row-local derived features for every column in ``FEATURE_SETS``.

    Raises ``ValueError`` if the input fails ``validate_input``.
    """
    problems = validate_input(frame)
    if problems:
        raise ValueError("Invalid feature input: " + "; ".join(problems))
    num = {c: pd.to_numeric(frame[c], errors="coerce") for c in SOURCE_NUMERIC}
    aa_ref, aa_alt = _text(frame["aa_ref"]), _text(frame["aa_alt"])
    missense = aa_ref.isin(HYDROPATHY.keys()) & aa_alt.isin(HYDROPATHY.keys())
    nt = _text(frame["nt_change"])
    codon = _text(frame["codon_ref"])

    out = pd.DataFrame(index=frame.index)
    out["cdna_position"] = num["cdna_position"]
    out["aa_position"] = num["aa_position"]
    out["exon_intron_number"] = _exon_intron_number(frame["exon_intron"])
    out["log_n_bp"] = np.log10(num["n_bp"].clip(lower=1))
    # Property changes are defined only for single amino-acid substitutions.
    out["hydropathy_change"] = (aa_alt.map(HYDROPATHY) - aa_ref.map(HYDROPATHY)).where(missense).astype(float)
    out["charge_change"] = (aa_alt.map(CHARGE) - aa_ref.map(CHARGE)).where(missense).astype(float)
    out["is_transition"] = nt.isin(TRANSITIONS).astype(float).where(nt.notna())
    out["ref_codon_has_cpg"] = codon.str.contains("cg", na=False, regex=False).astype(float).where(codon.notna())
    out["cys_gained"] = ((aa_alt == "cys") & (aa_ref != "cys")).astype(float).where(missense)
    out["cys_lost"] = ((aa_ref == "cys") & (aa_alt != "cys")).astype(float).where(missense)
    out["arg_lost"] = ((aa_ref == "arg") & (aa_alt != "arg")).astype(float).where(aa_ref.notna() & aa_alt.notna())
    out["log_fviii_activity"] = np.log10(num["fviii_activity"].clip(lower=0) + 0.1)
    out["log_fviii_antigen"] = np.log10(num["fviii_antigen"].clip(lower=0) + 0.1)

    for column in ("mut_type", "mut_effect", "domain", "location", "nt_change", "aa_ref",
                   "severity", "fviii_activity_kind", "fviii_antigen_kind", "crm_type"):
        out[column] = _text(frame[column])
    out["aa_alt_class"] = _aa_alt_class(frame["aa_alt"])
    # The FVIII parser marks absent values as kind "missing"; keep that as NaN
    # so imputation treats it the same way as any other missing category.
    for column in ("fviii_activity_kind", "fviii_antigen_kind"):
        out[column] = out[column].replace("missing", pd.NA)
    categorical = [c for c in out.columns if out[c].dtype == "string"]
    out[categorical] = out[categorical].astype(object).where(out[categorical].notna(), np.nan)
    return out
```

In [31]:
ENGINEERED = pd.DataFrame([
    ("log_n_bp", "n_bp", "log10(max(n_bp, 1))", "Variant length spans 1 to thousands of bp", "numeric", "Missing when n_bp is unknown"),
    ("exon_intron_number", "e_i_numb", "Whole numbers 1-26 kept; ranges, UTR labels and date serials -> missing",
     "Position along F8 at exon resolution", "numeric", "Exon and intron numbers share one scale"),
    ("hydropathy_change", "aa_first, aa_last", "Kyte-Doolittle(alt) - Kyte-Doolittle(ref)", "Physicochemical size of a substitution",
     "numeric", "Defined only for single amino-acid substitutions"),
    ("charge_change", "aa_first, aa_last", "charge(alt) - charge(ref); His counted neutral", "Gain or loss of side-chain charge",
     "numeric", "Substitutions only"),
    ("cys_gained / cys_lost", "aa_first, aa_last", "Indicator of a cysteine created or removed", "Cysteines form disulphide bonds",
     "numeric 0/1", "Substitutions only"),
    ("arg_lost", "aa_first, aa_last", "Indicator that arginine is replaced", "Arginine codons are mutation hotspots",
     "numeric 0/1", "Requires both amino acids"),
    ("ref_codon_has_cpg", "codon_first", "'cg' inside the reference codon", "CpG sites mutate more often",
     "numeric 0/1", "Misses CpGs spanning two codons"),
    ("is_transition", "ntchange", "A<->G or C<->T", "Mutation mechanism", "numeric 0/1", "Single-nucleotide changes only"),
    ("aa_alt_class", "aa_last", "amino acid / stop / frameshift / other", "Raw aa_last has 400+ text forms", "categorical", "Coarse"),
    ("log_fviii_activity", "clotting", "log10(FVIII:C + 0.1) after parsing ranges and bounds", "Residual activity on a usable scale",
     "numeric", "Bounds such as '<1' stored at the bound; kind kept separately"),
    ("log_fviii_antigen", "antigen", "log10(FVIII:Ag + 0.1)", "Antigen level", "numeric", "87% missing; mostly reflects reporting"),
    ("fviii_activity_kind / fviii_antigen_kind", "clotting / antigen", "exact / upper_bound / lower_bound / range_mid / missing",
     "Keeps how a value was reported", "categorical", "Can encode publication style"),
], columns=["feature", "source column(s)", "transformation", "why included", "output type", "limitation"])
display(ENGINEERED)
derived = features.derive_features(fused)
listed = {f.strip() for name in ENGINEERED["feature"] for f in name.split("/")}
assert listed <= set(derived.columns), listed - set(derived.columns)
print("Every engineered feature listed above exists in derive_features output.")
display(derived[[c for c in derived.columns if derived[c].dtype != object]].describe().T.round(3))

,feature,source column(s),transformation,why included,output type,limitation
0,log_n_bp,n_bp,"log10(max(n_bp, 1))",Variant length spans 1 to thousands of bp,numeric,Missing when n_bp is unknown
1,exon_intron_number,e_i_numb,"Whole numbers 1-26 kept; ranges, UTR labels and date serials -> missing",Position along F8 at exon resolution,numeric,Exon and intron numbers share one scale
2,hydropathy_change,"aa_first, aa_last",Kyte-Doolittle(alt) - Kyte-Doolittle(ref),Physicochemical size of a substitution,numeric,Defined only for single amino-acid substitutions
3,charge_change,"aa_first, aa_last",charge(alt) - charge(ref); His counted neutral,Gain or loss of side-chain charge,numeric,Substitutions only
4,cys_gained / cys_lost,"aa_first, aa_last",Indicator of a cysteine created or removed,Cysteines form disulphide bonds,numeric 0/1,Substitutions only
5,arg_lost,"aa_first, aa_last",Indicator that arginine is replaced,Arginine codons are mutation hotspots,numeric 0/1,Requires both amino acids
6,ref_codon_has_cpg,codon_first,'cg' inside the reference codon,CpG sites mutate more often,numeric 0/1,Misses CpGs spanning two codons
7,is_transition,ntchange,A<->G or C<->T,Mutation mechanism,numeric 0/1,Single-nucleotide changes only
8,aa_alt_class,aa_last,amino acid / stop / frameshift / other,Raw aa_last has 400+ text forms,categorical,Coarse
9,log_fviii_activity,clotting,log10(FVIII:C + 0.1) after parsing ranges and bounds,Residual activity on a usable scale,numeric,Bounds such as '<1' stored at the bound; kind kept separately


Every engineered feature listed above exists in derive_features output.


,count,mean,std,min,25%,50%,75%,max
cdna_position,4934.0,3574.138,2384.504,-1175.0,1407.25,3637.0,5981.0,7054.0
aa_position,4417.0,1232.963,776.498,-2.0,523.0,1213.0,2006.0,2352.0
exon_intron_number,4766.0,14.12,6.583,1.0,10.0,14.0,18.0,26.0
log_n_bp,4667.0,0.06,0.228,0.0,0.0,0.0,0.0,3.492
hydropathy_change,2725.0,1.024,3.789,-9.0,-2.2,1.0,3.8,9.0
charge_change,2725.0,-0.207,0.736,-2.0,-1.0,0.0,0.0,2.0
is_transition,3489.0,0.67,0.47,0.0,0.0,1.0,1.0,1.0
ref_codon_has_cpg,3279.0,0.322,0.467,0.0,0.0,0.0,1.0,1.0
cys_gained,2725.0,0.173,0.378,0.0,0.0,0.0,0.0,1.0
cys_lost,2725.0,0.019,0.136,0.0,0.0,0.0,0.0,1.0


Categorical columns pass through unchanged after spelling cleaning. These are `mut_type`, `mut_effect`,
`domain`, `location`, `nt_change`, `aa_ref`, `severity` and `crm_type`; together with the engineered
features they form the model inputs.

The five feature sets used in the project, and the number of encoded columns each produces:

In [32]:
feature_summary = json.loads((RESULTS / "features" / "feature_summary.json").read_text())
display(pd.DataFrame({name: {"numeric inputs": len(spec["numeric"]), "categorical inputs": len(spec["categorical"]),
                             "missing indicators": spec["missing_indicators"],
                             "encoded columns (saved)": feature_summary["sets"][name]["n_output_features"]}
                      for name, spec in features.FEATURE_SETS.items()}).T)

,numeric inputs,categorical inputs,missing indicators,encoded columns (saved)
genomic,11,7,True,92
clinical,2,4,True,23
combined,13,11,True,115
combined_minus_redundant,11,10,True,105
combined_no_reporting_flags,12,8,False,88


In [33]:
# Re-derive the combined encoding from the saved split (train rows only) and compare with the saved names.
from src.splitting import TRAIN, load_split

data = load_split()
pipeline = features.build_pipeline("combined").fit(data[data["split"] == TRAIN])
names = features.feature_name_map(pipeline)
saved_names = pd.read_csv(RESULTS / "features" / "feature_names_combined.csv")
assert list(names["output_feature"]) == list(saved_names["output_feature"])
print(f"Combined feature set: {len(names)} encoded columns, identical to results/features/feature_names_combined.csv")
display(names.groupby(["source_feature", "kind"]).size().unstack(fill_value=0))

Combined feature set: 115 encoded columns, identical to results/features/feature_names_combined.csv


kind,missing_indicator,numeric,one_hot
source_feature,,,
aa_alt_class,0,0,5
aa_position,1,1,0
aa_ref,0,0,21
arg_lost,1,1,0
cdna_position,1,1,0
charge_change,1,1,0
crm_type,0,0,3
cys_gained,1,1,0
cys_lost,1,1,0


This refit is only the feature encoder (imputation statistics, scaling and category lists). No model
is trained. It matches the saved encoder column for column.

## 8. Final cleaned and combined dataset

In [34]:
schema = json.loads((PROCESSED / "feature_schema.json").read_text())
print("data/processed/fused_reports.csv:", fused.shape[0], "rows x", fused.shape[1], "columns")
display(pd.DataFrame({
    "columns": [", ".join(schema["genomic"]), ", ".join(schema["clinical"]), schema["target"],
                ", ".join(c for c in schema["non_predictors"] if c != schema["target"])],
}, index=["genomic (MMC2)", "clinical (MMC3)", "target", "identifiers, grouping and metadata (never predictors)"]))

data/processed/fused_reports.csv: 4962 rows x 32 columns


,columns
genomic (MMC2),"mut_type, mut_effect, domain, location, exon_intron, aa_position, cdna_position, n_bp,..."
clinical (MMC3),"severity, fviii_activity, fviii_activity_kind, fviii_antigen, fviii_antigen_kind, crm_..."
target,y
"identifiers, grouping and metadata (never predictors)","all_id, mut_id, variant_key, label_status, gene, hgvs_cdna, hgvs_protein, annotation, ..."


In [35]:
miss = fused[schema["genomic"] + schema["clinical"]].isna().mean().mul(100).round(2)
display(miss.sort_values(ascending=False).to_frame("% missing"))

,% missing
crm_type,92.77
fviii_antigen,86.98
codon_alt,33.98
codon_ref,33.92
nt_change,29.69
fviii_activity,17.09
aa_ref,11.63
aa_alt,11.59
aa_position,10.98
n_bp,5.95


In [36]:
dist = fused["y"].value_counts().rename(index={0: "no inhibitor (0)", 1: "inhibitor (1)"})
display(pd.DataFrame({"reports": dist, "share": (dist / dist.sum()).round(4)}))
print(f"mut_id values: {fused['mut_id'].nunique():,}; variant groups (variant_key): {fused[GROUP_KEY].nunique():,}")

,reports,share
y,,
no inhibitor (0),4126,0.832
inhibitor (1),836,0.169


mut_id values: 2,639; variant groups (variant_key): 2,473


A small sample follows, showing predictors, target and grouping key only. It leaves out publication
fields, HGVS names and curator text.

In [37]:
sample_cols = ["all_id", GROUP_KEY, "mut_type", "mut_effect", "domain", "exon_intron", "aa_ref", "aa_alt",
               "severity", "fviii_activity", "fviii_activity_kind", "y"]
display(fused[sample_cols].sample(6, random_state=2026).reset_index(drop=True))

,all_id,variant_key,mut_type,mut_effect,domain,exon_intron,aa_ref,aa_alt,severity,fviii_activity,fviii_activity_kind,y
0,6772,v3414,deletion,large deletion,multiple domains,7--8,NaN,NaN,severe,1.0,upper_bound,1
1,3257,v1307,point,missense,a3,16,glu,lys,severe,1.0,upper_bound,0
2,2740,v1071,point,nonsense,b,14,tyr,*,severe,1.0,upper_bound,0
3,4006,v1594,point,missense,c1,22,tyr,cys,moderate,NaN,missing,0
4,265,v999004,point,missense,a1,4,glu,asp,mild,8.0,exact,0
5,7813,v4455,point,missense,a3,15,phe,ser,severe,NaN,missing,0


The positive rate (16.85%) is the share of **published** reports with inhibitors. It is not an
incidence, because some publications report only inhibitor patients (Section 5.1).

## 9. Validation checks

Each check below runs against the implementation and the saved artifacts, and the results are
collected in one table. The headline figures quoted in the project summary are verified here, not
assumed: 4,962 reports, 836 positives, 2,473 variant groups and 115 combined encoded columns.

In [38]:
from src.splitting import DISJOINT_KEYS, SPLITS, assert_disjoint

checks = []


def record(name, passed, detail):
    checks.append({"check": name, "passed": bool(passed), "detail": detail})


record("target values", set(fused["y"].unique()) == {0, 1}, f"values {sorted(int(v) for v in fused['y'].unique())}")
record("only confirmed labels kept", set(fused["label_status"]) == {labels.CONFIRMED_POSITIVE, labels.CONFIRMED_NEGATIVE},
       str(fused["label_status"].value_counts().to_dict()))
record("unknown labels never negative", mapping.loc[unknown, "y"].isna().all(),
       f"{int(unknown.sum())} unknown/invalid labels mapped to missing")
record("report ids unique", fused["all_id"].is_unique, f"{fused['all_id'].nunique():,} distinct all_id")
record("merge cardinality", len(merged_now) == len(kept), f"{len(kept):,} rows before, {len(merged_now):,} after")
record("no missing identifiers", fused[["all_id", "mut_id", GROUP_KEY]].notna().all().all(), "all_id, mut_id, variant_key present")
record("every report has an F8 variant", (fused["gene"] == pp.F8_GENE).all(), "gene == f8 for all rows")
record("one variant_key per mut_id", (fused.groupby("mut_id")[GROUP_KEY].nunique() == 1).all(), "union-find key is consistent")
hgvs_ok = all((fused.dropna(subset=[c]).groupby(c)[GROUP_KEY].nunique() == 1).all() for c in ("hgvs_cdna", "hgvs_protein"))
record("each HGVS name in one group", hgvs_ok, "cDNA and protein names never span two variant_keys")
predictors = schema["genomic"] + schema["clinical"]
record("no target or proxy among predictors", not (set(predictors) & FORBIDDEN_PREDICTORS), f"{len(predictors)} candidate predictors checked")
record("final schema matches fusion constants",
       schema["genomic"] == list(GENOMIC_COLUMNS) and schema["clinical"] == list(CLINICAL_COLUMNS), "feature_schema.json")
record("encoded columns contain no forbidden source",
       not (set(names["source_feature"]) & FORBIDDEN_PREDICTORS), f"{len(names)} combined columns")
assert_disjoint(data, data["split"])
record("variant groups never span partitions", True, f"checked {', '.join(DISJOINT_KEYS)} across {', '.join(SPLITS)}")
record("split covers every report", len(data) == len(fused) and data["split"].isin(SPLITS).all(), f"{len(data):,} rows assigned")

fresh = fusion.run_fusion(CONFIG)  # in memory; writes nothing
pd.testing.assert_frame_equal(fresh.reports.reset_index(drop=True), fused, check_dtype=False)
record("in-memory rerun reproduces fused_reports.csv", True, "pandas assert_frame_equal")

expected = {"reports": 4962, "positives": 836, "variant groups": 2473, "combined encoded columns": 115}
observed = {"reports": len(fused), "positives": int(fused["y"].sum()),
            "variant groups": fused[GROUP_KEY].nunique(), "combined encoded columns": len(names)}
record("headline figures", observed == expected, str(observed))

result = pd.DataFrame(checks)
display(result)
assert result["passed"].all(), result[~result["passed"]]
print(f"{len(result)}/{len(result)} checks passed.")

,check,passed,detail
0,target values,True,"values [0, 1]"
1,only confirmed labels kept,True,"{'confirmed_negative': 4126, 'confirmed_positive': 836}"
2,unknown labels never negative,True,5098 unknown/invalid labels mapped to missing
3,report ids unique,True,"4,962 distinct all_id"
4,merge cardinality,True,"4,962 rows before, 4,962 after"
5,no missing identifiers,True,"all_id, mut_id, variant_key present"
6,every report has an F8 variant,True,gene == f8 for all rows
7,one variant_key per mut_id,True,union-find key is consistent
8,each HGVS name in one group,True,cDNA and protein names never span two variant_keys
9,no target or proxy among predictors,True,19 candidate predictors checked


16/16 checks passed.


## 10. Final summary

In [39]:
final_row = flow.iloc[-1]
n_genomic = int((selection["final role"] == "Genomic predictors").sum())
n_clinical = int((selection["final role"] == "Clinical predictors").sum())
n_excluded = int((selection["decision"] == "Excluded").sum())
display(Markdown(f"""
**Preparation and combination.**
- MMC2 ({mmc2.shape[0]:,} variants × {mmc2.shape[1]} columns) and MMC3 ({mmc3.shape[0]:,} reports ×
  {mmc3.shape[1]} columns) were typed and cleaned with the project's own functions.
- Spelling variants and sentinel tokens were normalised, FVIII values parsed, and the outcome mapped
  with unknown values kept as missing.
- Report rows were removed in a fixed, counted order: shifted rows, gnomAD population rows, a report
  with no variant record, F9 variants, and unknown outcomes.
- The remaining reports were joined many-to-one to their variant on `mut_id`. The join created no
  duplicate rows.
- Variants sharing an HGVS name were merged into a `variant_key` so they can never span two partitions.

**Result.** {int(final_row['rows']):,} labelled reports ({int(final_row['positives'])} with inhibitors,
{100 * final_row['positives'] / final_row['rows']:.2f}%) from {fused['mut_id'].nunique():,} `mut_id`s in
{fused[GROUP_KEY].nunique():,} variant groups. There are {len(schema['genomic'])} genomic and
{len(schema['clinical'])} clinical candidate columns, which the feature pipeline turns into {len(names)}
encoded model inputs (combined set).

**Selected.** {n_genomic} original MMC2 parameters describing the variant (type, effect, domain,
location, exon, positions, length, nucleotide and amino-acid change) and {n_clinical} original MMC3
parameters describing the patient (severity, FVIII:C, FVIII:Ag, CRM status). Engineered features add
substitution properties, mutation-mechanism flags and log-scaled levels.

**Excluded ({n_excluded} original parameters).**
- The variant-level outcome summaries (`uinhibitor` and the other `u*` fields), as target proxies.
- `Count_mut_id`, which is stale.
- Publication, laboratory and patient-country fields, which carry selection bias.
- Curator comments, which reflect curation batch rather than biology.
- Patient codes and free text.
- Constant or empty columns.

Identifiers, HGVS names (used only to define variant groups) and publication metadata are kept in
the dataset for traceability but are never model inputs.
"""))


**Preparation and combination.**
- MMC2 (6,211 variants × 34 columns) and MMC3 (10,064 reports ×
  23 columns) were typed and cleaned with the project's own functions.
- Spelling variants and sentinel tokens were normalised, FVIII values parsed, and the outcome mapped
  with unknown values kept as missing.
- Report rows were removed in a fixed, counted order: shifted rows, gnomAD population rows, a report
  with no variant record, F9 variants, and unknown outcomes.
- The remaining reports were joined many-to-one to their variant on `mut_id`. The join created no
  duplicate rows.
- Variants sharing an HGVS name were merged into a `variant_key` so they can never span two partitions.

**Result.** 4,962 labelled reports (836 with inhibitors,
16.85%) from 2,639 `mut_id`s in
2,473 variant groups. There are 13 genomic and
6 clinical candidate columns, which the feature pipeline turns into 115
encoded model inputs (combined set).

**Selected.** 12 original MMC2 parameters describing the variant (type, effect, domain,
location, exon, positions, length, nucleotide and amino-acid change) and 4 original MMC3
parameters describing the patient (severity, FVIII:C, FVIII:Ag, CRM status). Engineered features add
substitution properties, mutation-mechanism flags and log-scaled levels.

**Excluded (30 original parameters).**
- The variant-level outcome summaries (`uinhibitor` and the other `u*` fields), as target proxies.
- `Count_mut_id`, which is stale.
- Publication, laboratory and patient-country fields, which carry selection bias.
- Curator comments, which reflect curation batch rather than biology.
- Patient codes and free text.
- Constant or empty columns.

Identifiers, HGVS names (used only to define variant groups) and publication metadata are kept in
the dataset for traceability but are never model inputs.


**Data-quality issues and limitations**

- **Publication-driven sampling.** Some papers report only inhibitor patients. The positive rate is
  therefore not an incidence, and whether a field was reported partly reflects the publication.
- **Missing genotypes.** Intron 22 and intron 1 inversions, the commonest severe genotypes, are absent.
- **Missing clinical factors.** Exposure days, treatment, product, age, HLA and family history are not
  available.
- **Sparse clinical fields.** FVIII:Ag (about 87% missing) and CRM status (about 93% missing) are mostly
  empty.
- **Exon date serials.** 69 reports carry exon values corrupted into date serials; they are treated as
  missing.
- **Possible repeated patients.** Identical reports and repeated `(ref_id, pa_id)` codes are kept as
  separate reports.

**Unresolved questions (for review)**

1. The source journal, database version, acquisition date and licence of MMC2/MMC3 are not verified.
2. Column interpretations in Section 3 are inferred from names and values, not from source documentation.
3. Whether the 5-digit exon values were originally ranges, and which ones, cannot be confirmed from the data.
4. Whether reports sharing a `(ref_id, pa_id)` code, or identical in every field, are the same patient
   is unknown. The implementation keeps them as separate reports.